# NB5 — Merge + Deploy + GGUF

**Stack:** Unsloth `merge_and_unload` + `save_pretrained_gguf(quantization='Q4_K_M')`
+ llama-cpp-python smoke test.
Maps to deck §7.1 lab brief: "merge adapter, quantize GGUF, serve với vLLM".

> **Mục tiêu:** export the SFT+DPO adapter as a deployable GGUF Q4_K_M file
> (~1.5 GB on 3B / ~4 GB on 7B), then smoke-test it through llama-cpp-python.
> Final cell shows the optional vLLM serving command (BigGPU only).

## 0. Setup

In [1]:
import os
import json
from pathlib import Path

COMPUTE_TIER = os.environ.get("COMPUTE_TIER", "T4").upper()
BASE_MODEL = (
    "unsloth/Qwen2.5-3B" if COMPUTE_TIER == "T4"
    else "unsloth/Qwen2.5-7B-bnb-4bit"
)
MAX_LEN = 512 if COMPUTE_TIER == "T4" else 1024

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DPO_PATH = REPO_ROOT / "adapters" / "dpo"
MERGED_PATH = REPO_ROOT / "adapters" / "merged-fp16"
GGUF_DIR = REPO_ROOT / "gguf"
MERGED_PATH.mkdir(parents=True, exist_ok=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)

assert DPO_PATH.exists(), "NB3 must run first"

print(f"COMPUTE_TIER:    {COMPUTE_TIER}")
print(f"DPO adapter:     {DPO_PATH}")
print(f"merged output:   {MERGED_PATH}")
print(f"GGUF output:     {GGUF_DIR}")

COMPUTE_TIER:    T4
DPO adapter:     /home/quang_ai/admin/AIThucChien/Phase2-Track3/DPO-Alignment/adapters/dpo
merged output:   /home/quang_ai/admin/AIThucChien/Phase2-Track3/DPO-Alignment/adapters/merged-fp16
GGUF output:     /home/quang_ai/admin/AIThucChien/Phase2-Track3/DPO-Alignment/gguf


In [2]:
import torch

assert torch.cuda.is_available()

## 1. Load DPO model + merge adapter

In [3]:
from unsloth import FastLanguageModel
from peft import PeftModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LEN,
    dtype=None,
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Stack SFT-mini → DPO adapters
SFT_PATH = REPO_ROOT / "adapters" / "sft-mini"
model = PeftModel.from_pretrained(model, str(SFT_PATH))
print(f"Loaded SFT-mini adapter from {SFT_PATH}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/quang_ai/miniconda3/envs/aip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/quang_ai/miniconda3/envs/aip/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.1)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Ti. Num GPUs = 1. Max memory: 15.996 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 434/434 [00:01<00:00, 385.73it/s]


unsloth/qwen2.5-3b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded SFT-mini adapter from /home/quang_ai/admin/AIThucChien/Phase2-Track3/DPO-Alignment/adapters/sft-mini


> **Note:** The DPO adapter trained in NB3 stacks on top of SFT. To get a fully
> aligned merged model, we apply both adapters before merging. Unsloth's
> `save_pretrained_merged` handles the SFT + DPO + base merge in one shot.

## 2. Save merged FP16 weights

`save_pretrained_merged(method="merged_16bit")` produces a HuggingFace-format
directory you can either upload to HF Hub directly OR feed into the GGUF
converter in step 3.

In [6]:
# Clear GPU cache first
import torch, gc
gc.collect()
torch.cuda.empty_cache()

model.save_pretrained_merged(
    str(MERGED_PATH),
    tokenizer,
    save_method="merged_16bit",
)

# Remove leftover quantization_config from config.json
import json as _json
cfg_path = MERGED_PATH / "config.json"
cfg = _json.loads(cfg_path.read_text())
cfg.pop("quantization_config", None)
cfg_path.write_text(_json.dumps(cfg, indent=2))

print(f"Saved clean merged FP16 to {MERGED_PATH}")

del model
gc.collect()
torch.cuda.empty_cache()


Unsloth: Saving full fine-tuned model to '/home/quang_ai/admin/AIThucChien/Phase2-Track3/DPO-Alignment/adapters/merged-fp16' ...


NotImplementedError: 

## 3. Quantize to GGUF Q4_K_M

Q4_K_M is the sweet spot: ~4× compression vs FP16, minimal quality loss.
Unsloth wraps llama.cpp's `quantize` binary — first run downloads + compiles
llama.cpp (~3 min) then quantizes (~30 s).

In [ ]:
# Reload the merged model — Unsloth's GGUF saver expects a live model handle.
from unsloth import FastLanguageModel as FLM

model, tokenizer = FLM.from_pretrained(
    model_name=str(MERGED_PATH),
    load_in_4bit=False,
)

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4060 Ti. Num GPUs = 1. Max memory: 15.996 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Exception: Unsloth: Critical error since some weights are not initialized.
Please try updating Unsloth, transformers and timm via:
`pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo transformers timm`
<LogRecord: transformers.modeling_utils, 30, /home/quang_ai/miniconda3/envs/aip/lib/python3.10/site-packages/transformers/modeling_utils.py, 5525, "Some weights of the model checkpoint at /home/quang_ai/admin/AIThucChien/Phase2-Track3/DPO-Alignment/adapters/merged-fp16 were not used when initializing Qwen2ForCausalLM: ['model.layers.0.mlp.down_proj.base_layer.weight', 'model.layers.0.mlp.down_proj.base_layer.weight.absmax', 'model.layers.0.mlp.down_proj.base_layer.weight.nested_absmax', 'model.layers.0.mlp.down_proj.base_layer.weight.nested_quant_map', 'model.layers.0.mlp.down_proj.base_layer.weight.quant_map', 'model.layers.0.mlp.down_proj.base_layer.weight.quant_state.bitsandbytes__nf4', 'model.layers.0.mlp.down_proj.lora_A.default.weight', 'model.laye

In [ ]:
# Save GGUF in 1 quantization tier (Q4_K_M). Add more tiers below if you want the
# +3 "GGUF release published" rigor add-on.
model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"Saved GGUF Q4_K_M to {GGUF_DIR}")

### 3a. Optional — additional quantization tiers (for the +3 rigor add-on)

In [ ]:
# Uncomment if you want Q5_K_M + Q8_0 too (~2× total disk space).
# Each adds ~30s for an extra GGUF file.
#
# model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method="q5_k_m")
# model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method="q8_0")

In [ ]:
import os

print("GGUF files:")
for p in sorted(GGUF_DIR.iterdir()):
    if p.suffix == ".gguf":
        size_mb = p.stat().st_size / 1e6
        print(f"  {p.name:50s}  {size_mb:>8.1f} MB")

del model
gc.collect()
torch.cuda.empty_cache()

## 4. Smoke test with llama-cpp-python

In [ ]:
from llama_cpp import Llama

# Find the Q4_K_M GGUF
gguf_files = list(GGUF_DIR.glob("*Q4_K_M*.gguf")) + list(GGUF_DIR.glob("*q4_k_m*.gguf"))
assert gguf_files, "No Q4_K_M GGUF found — step 3 may have failed"
gguf_path = gguf_files[0]
print(f"Loading: {gguf_path.name}")

# n_gpu_layers=-1 offloads all layers to GPU if compiled with CUDA/Metal/Vulkan
llm = Llama(
    model_path=str(gguf_path),
    n_ctx=MAX_LEN,
    n_gpu_layers=-1,           # all layers on GPU; falls back to CPU if no GPU compile
    verbose=False,
)
print("Loaded.")

### 4a. Smoke prompt + response (deliverable: `06-gguf-smoke.png`)

In [ ]:
SMOKE_PROMPT = "Giải thích ngắn gọn (3 câu) cách thuật toán Bubble sort hoạt động."

response = llm.create_chat_completion(
    messages=[{"role": "user", "content": SMOKE_PROMPT}],
    max_tokens=200,
    temperature=0.0,
)

print(f"PROMPT:\n  {SMOKE_PROMPT}\n")
print(f"RESPONSE (Q4_K_M GGUF, llama-cpp-python):\n  {response['choices'][0]['message']['content']}")
print(f"\nTokens used: {response['usage']}")

## 5. Optional — vLLM serving (BigGPU only)

vLLM provides production-grade OpenAI-compatible serving. **Requires CUDA GPU
with ≥ 16 GB VRAM** and `vllm` installed (see `requirements-biggpu.txt`).
On T4 tier this cell will OOM. Skip on T4.

Run in a SEPARATE terminal (NOT in the notebook — vLLM blocks until killed):

```bash
pip install vllm                         # once
vllm serve adapters/merged-fp16 \
  --port 8000 \
  --max-model-len 1024 \
  --gpu-memory-utilization 0.9
```

Then test:

```bash
curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "merged-fp16", "messages": [{"role": "user", "content": "Hello"}]}'
```

**Why not in the notebook?** vLLM's process model doesn't play nicely with
Jupyter — it expects to own the GPU + a long-running HTTP server. Run it as
a sidecar process. The deck mentions vLLM as the deploy target; for actual
production you'd containerize this command. For the lab, llama-cpp-python in
step 4 is the graded artifact.

## 6. Save deployment metadata

In [ ]:
deploy_meta = {
    "compute_tier": COMPUTE_TIER,
    "base_model": BASE_MODEL,
    "merged_path": str(MERGED_PATH),
    "gguf_path": str(gguf_path),
    "gguf_size_mb": round(gguf_path.stat().st_size / 1e6, 1),
    "quantization": "q4_k_m",
    "smoke_prompt": SMOKE_PROMPT,
    "smoke_response": response["choices"][0]["message"]["content"],
}
(REPO_ROOT / "data" / "eval" / "deploy_meta.json").parent.mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "data" / "eval" / "deploy_meta.json").write_text(
    json.dumps(deploy_meta, ensure_ascii=False, indent=2)
)
print("Saved data/eval/deploy_meta.json")

## 7. Submission checklist

Bạn vừa hoàn thành core lab. Trước khi submit:

1. **Run** `make verify` — gatekeeper sẽ list missing artifacts.
2. **Take screenshots** vào `submission/screenshots/` (xem `submission/screenshots/README.md`).
3. **Fill** `submission/REFLECTION.md` — đặc biệt là § 3 (reward curves analysis,
   cross-reference deck §3.4) và § 6 (single change that mattered most).
4. **(Optional)** Pick a rigor add-on từ rubric.md (β-sweep, HF push, GGUF
   release, W&B link, cross-judge).
5. **(Optional)** Pick a `BONUS-CHALLENGE.md` provocation cho creative bonus.

Push public repo + paste URL vào VinUni LMS Day-22 box.

Câu hỏi cuối để brainstorm trước khi đóng laptop:

> **The deck says:** "DPO + 30 min A100 + 2k UltraFeedback → 3.2 → 4.1 helpfulness."
> **You measured:** _<your win-rate from NB4>_.
> **Why might they differ?** Dataset (English vs VN), base model (Qwen2.5-3B vs
> deck's unspecified base), judge bias, sample size (8 prompts vs deck's full eval).
> Đó chính là § 6 trong REFLECTION — what 1 change would close the gap.